#### This code authenticates and initializes the Google Earth Engine API and installs the earthengine-api and geemap libraries for visualizing geographical data in Jupyter Notebooks.

In [ ]:
#!pip install geemap

In [ ]:
!pip install earthengine-api

In [ ]:
import geemap
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project='my-project')

#### Part 1: This dataset represents the watershed boundaries across the United States, based on the USGS Watershed Boundary Dataset (WBD). Watersheds, also known as drainage basins, define areas where water collects and flows into a common outlet, such as a river or lake.

In [ ]:
# Load the USGS Watershed Boundary Dataset (HUC-4 level)
watershed = ee.FeatureCollection("USGS/WBD/2017/HUC04")

# Define the style: blue border with no fill
style = {
    "color": "0000FF",  # Blue border
    "width": 2,         # Border width (2 pixels)
    "fillColor": "00000000"  # Transparent fill (no fill)
}

# Create the map
Map = geemap.Map()

# Add the watershed boundary layer with the defined style
Map.addLayer(watershed.style(**style), {}, "Watershed Boundaries")

# Center the map on the U.S. with an appropriate zoom level
Map.setCenter(-98.5795, 39.8283, 5)

# Add a text label to the map
Map.add_text("Made by Clara MR", fontsize=20, position='bottomright')

# Display the map
Map

#### Part 2: This code creates a map centered on the intersected watershed in Utah, showing the state boundary, all watershed boundaries, and the intersected watershed with appropriate styles for each layer.

In [ ]:
# Load the counties dataset from the US Census data (or any other relevant dataset)
counties = ee.FeatureCollection("TIGER/2018/Counties")

# Filter for the county of your choice in Utah (e.g., Salt Lake County)
utah_county = counties.filter(ee.Filter.And(
    ee.Filter.eq('STATEFP', '49'),  # Utah's state FIPS code
    ee.Filter.eq('COUNTYFP', '035')  # Salt Lake County FIPS code
))

# Load the TIGER/2018 states dataset and filter for the state of Utah
utah_state = ee.FeatureCollection("TIGER/2018/States").filter(ee.Filter.eq('STUSPS', 'UT'))

# Select the watershed(s) that intersect with the chosen county
intersected_watershed = watershed.filterBounds(utah_county)

# Create a new map for the intersection
Map_intersection = geemap.Map()

# Define the styles
watershed_style = {
    'color': '#000000',  # Border color (black)
    'width': 2,         # Border width
    'fillColor': '#80808066'  # Gray color with 40% transparency in hexadecimal format
}

state_style = {
    'color': 'FF0000',  # Border color (red)
    'width': 2,         # Border width
    'fillColor': '00000000'  # Transparent fill
}

# Add the state boundary and intersected watershed layers to the map
Map_intersection.addLayer(watershed.style(**style), {}, "Watershed Boundaries")
Map_intersection.addLayer(utah_state.style(**state_style), {}, "Utah State Boundary")
Map_intersection.addLayer(intersected_watershed.style(**watershed_style), {}, "Intersected Watershed")

# Center the map on the intersected watershed
Map_intersection.centerObject(intersected_watershed, zoom=7)  # Adjust the zoom level as needed

# Add a text label to the map
Map_intersection.add_text("Made by Clara MR", fontsize=20, position='bottomright')

# Display the new map
Map_intersection


#### Part 3: This code clips the DEM obtained from the USGS 3DEP 10m dataset to this basin, and displays it with a color palette suitable for elevation visualization.

In [ ]:
# Create a new map for the DEM intersection
Map_dem_intersection = geemap.Map()

# Add the intersected watershed layer to the new map with the defined style
Map_dem_intersection.addLayer(intersected_watershed.style(**watershed_style), {}, "Intersected Watershed")

# Load the DEM dataset
dem = ee.Image("USGS/3DEP/10m")

# Clip the DEM to the intersected watershed
clipped_dem = dem.clip(intersected_watershed)

# Define the visualization parameters for the DEM
dem_vis_params = {
    'min': 000,
    'max': 3000,
    'palette': [ '0000FF',  # Blue
        '00FFFF',  # Cyan
        '00FF00',  # Green
        'FFFF00',  # Yellow
        'FFA500',  # Orange
        'FF0000',  # Red
        '800080',  # Purple
        'FFFFFF'   # White
]
}

# Add the clipped DEM to the new map with the defined visualization parameters
Map_dem_intersection.addLayer(clipped_dem, dem_vis_params, "Clipped DEM")

# Add a color bar to the new map
Map_dem_intersection.add_colorbar(dem_vis_params['palette'], vmin=dem_vis_params['min'], vmax=dem_vis_params['max'], label="Elevation (m)")

# Center the new map on the intersected watershed
Map_dem_intersection.centerObject(intersected_watershed,zoom=7)  # Adjust the zoom level as needed

# Add a text label to the new map
Map_dem_intersection.add_text("Made by Clara MR", fontsize=20, position='bottomright')

# Display the new map
Map_dem_intersection

#### Part 4: This code creates a split-panel map to visualize land cover change between 2001 and 2019 for the state of Utah, using the USGS NLCD dataset and adding a legend and text labels for clarity.

In [ ]:
# Load the NLCD datasets for 2001 and 2019
nlcd = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD")
nlcd_2001 = nlcd.filter(ee.Filter.eq('system:index', '2001')).first().select('landcover')
nlcd_2019 = nlcd.filter(ee.Filter.eq('system:index', '2019')).first().select('landcover')

# Clip the NLCD datasets to the state of Utah
nlcd_2001_clipped = nlcd_2001.clip(utah_state)
nlcd_2019_clipped = nlcd_2019.clip(utah_state)

# Define the visualization parameters for NLCD
nlcd_vis_params = {
    'min': 0,
    'max': 95,
    'palette': [
        '466b9f', 'd1def8', 'dec5c5', 'd99282', 'eb0000', 'ab0000', 'b3ac9f', '68ab5f',
        '1c5f2c', 'b5ca8f', 'a3cc51', '82ba9e', 'dcd93d', 'ab7028', 'bad9eb', '70a3ba',
        'e9e9e9', '6c9fb8'
    ]
}

# Create a split-panel map
Map = geemap.Map()

# Add the NLCD 2001 layer to the left panel
left_layer = geemap.ee_tile_layer(nlcd_2001_clipped, nlcd_vis_params, 'NLCD 2001')

# Add the NLCD 2019 layer to the right panel
right_layer = geemap.ee_tile_layer(nlcd_2019_clipped, nlcd_vis_params, 'NLCD 2019')

# Set the split panel
Map.split_map(left_layer, right_layer)

# Add the NLCD legend to the map
Map.add_legend(builtin_legend='NLCD', title='NLCD Land Cover Type')

# Center the map on the state of Utah
Map.centerObject(utah_state, 6)

# Add a text label to the map
Map.add_text("Land Cover Change (2001-2019)", fontsize=15, position='bottomright')
Map.add_text("Made by Clara MR", fontsize=10, position='bottomright', offset=[0, 25])

# Display the map
Map

#### Part 5: The code retrieves restaurants in Salt Lake City from OpenStreetMap, handles both Point and Polygon geometries by calculating the centroid for Polygons, and visualizes them on an interactive map using Geemap and GeoPandas.

In [ ]:
!pip install geopandas osmnx geemap

In [ ]:
import geemap.osm as osm

In [ ]:
import osmnx as ox
import geopandas as gpd
import folium
import matplotlib.pyplot as plt

In [ ]:
# Create an interactive map with Geemap
Map = geemap.Map()

# Geocode Salt Lake City and obtain its geometries (GeoDataFrame)
gdf = ox.geocode_to_gdf("Salt Lake City")

# Convert the GeoDataFrame to an Earth Engine FeatureCollection
fc = geemap.gdf_to_ee(gdf)

# Add the Salt Lake City geometry to the map (as Earth Engine FeatureCollection)
Map.add_ee_layer(fc, {}, "Salt Lake City (EE)")

# Download restaurants from OSM within Salt Lake City (GeoDataFrame)
# We filter places by the amenity tag: 'restaurant'
restaurants = ox.features_from_place("Salt Lake City, Utah, USA", tags={"amenity": "restaurant"})

# Function to handle both Point and Polygon geometries
def handle_geometry(feature):
    # Check if the geometry is a Point or not
    if feature['geometry'].geom_type == 'Point':
        return feature['geometry']  # Return the Point directly
    else:
        # If it's a Polygon or MultiPolygon, return the centroid
        return feature['geometry'].centroid

# Apply the function to all restaurants to ensure we have Points
restaurants['geometry'] = restaurants.apply(handle_geometry, axis=1)

# Add the restaurants to the map with a red color
Map.add_gdf(restaurants, layer_name="Restaurants", style={'color': 'red'})

# Center the map on the state of Utah
Map.centerObject(fc, 12)

# Display the map
Map
